# Simple ARC API call
Use the HTML interface to pass a question and ARC API key. The notebook reads `inputs` and returns `outputs`. Keys are not embedded in this notebook.


In [ ]:
import json
from pyodide.http import pyfetch

async def ask_llm(api_key, question):
    api_key = api_key.strip()
    question = question.strip() or "Why is the sky blue?"
    if not api_key:
        raise ValueError("Enter your ARC API key in the HTML form.")
    print("Thinking...")
    try:
        response = await pyfetch(
            "https://llm-api.arc.vt.edu/api/v1/chat/completions",
            method="POST",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            body=json.dumps({
                "model": "gpt-oss-120b",
                "messages": [
                    {"role": "system", "content": "Answer briefly in plain language."},
                    {"role": "user", "content": question},
                ],
            }),
        )
        if not response.ok:
            raise RuntimeError(f"Request failed (HTTP {response.status}). Check your API key and model.")
        result = await response.json()
        answer = result["choices"][0]["message"]["content"]
        return {"answer": answer}
    except OSError as error:
        raise RuntimeError("Browser request failed. Check your connection and whether ARC permits requests from this GitHub Pages site (CORS).") from error

# The HTML bridge supplies inputs. Running directly in JupyterLite still works.
if "inputs" not in globals():
    import getpass
    inputs = {
        "api_key": getpass.getpass("Enter your ARC API key (hidden): "),
        "question": input("Ask a question [Why is the sky blue?]: "),
    }

try:
    outputs = await ask_llm(inputs.get("api_key", ""), inputs.get("question", ""))
    print(outputs["answer"])
finally:
    inputs.pop("api_key", None)
